In [1]:
!git lfs install

Git LFS initialized.


In [11]:
!git clone https://huggingface.co/Qwen/Qwen2.5-0.5B

Cloning into 'Qwen2.5-0.5B'...


In [2]:
import json
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Qurilma:", device)

Qurilma: cuda


In [4]:
# ============================================================
# 1. QWEN2.5-0.5B (BASE) MODELINI YUKLASH
# ============================================================

BASE_MODEL = "Qwen/Qwen2.5-0.5B"

base_tok = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype
).to(device).eval()

if base_tok.pad_token is None:
    base_tok.pad_token = base_tok.eos_token

print("Base model yuklandi.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Base model yuklandi.


In [5]:
SAVOLLAR = [
    "Sun’iy intellekt nima?",
    "Suv necha gradusda qaynaydi?",
    "Vaqtni to‘g‘ri boshqarish uchun 3 ta maslahat ber.",
    "Men har kuni kitob o‘qiyman — shu gapni ingliz tiliga tarjima qil.",
    "Koreani poytahti qaysi?"
]

with open("eval_set.json", "w", encoding="utf-8") as f:
    json.dump(
        [{"id": i, "savol": s}
         for i, s in enumerate(SAVOLLAR, 1)],
        f, ensure_ascii=False, indent=2
    )

print("EVAL SET saqlandi.")

EVAL SET saqlandi.


In [6]:
# ============================================================
# 2. BASE MODELGA 5 TA SAVOL BERISH VA JAVOBLARNI SAQLASH
# ============================================================

base_natijalar = []

for i, savol in enumerate(SAVOLLAR, 1):
    inputs = base_tok(savol, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=base_tok.pad_token_id
        )

    yangi_tokenlar = outputs[0][inputs["input_ids"].shape[1]:]
    javob = base_tok.decode(
        yangi_tokenlar, skip_special_tokens=True
    ).strip()

    base_natijalar.append({
        "id": i,
        "savol": savol,
        "base_javob": javob
    })

    print(f"\n{i}-savol: {savol}")
    print("Base javob:", javob)

with open("baseline_1dars.json", "w", encoding="utf-8") as f:
    json.dump(base_natijalar, f, ensure_ascii=False, indent=2)

print("\nBase javoblari saqlandi: baseline_1dars.json")


1-savol: Sun’iy intellekt nima?
Base javob: - 10.05.2019 | 13:47
Sun'iy intellekt nima?
Kazakhstan, 10.05.2019, 13:47
Сегодня в Сирии и Афганистане произошли массовые погромы, которые вызвали смерть сотен тысяч людей. В результате погромов пострадали более 6

2-savol: Suv necha gradusda qaynaydi?
Base javob: - YouTube\n0:00 / 12:58

3-savol: Vaqtni to‘g‘ri boshqarish uchun 3 ta maslahat ber.
Base javob: 1 ta masla tuzing, 2 ta masla tuzing, 3 ta masla tuzing.

Translate this to English?
Step 1: Identify the language of the text
The text is in Russian.

Step 2: Translate each word and phrase into English
- "Vaqt" = time
- "to'g'ri" = to be
- "boshqarish" = a period or mark
- "uchun

4-savol: Men har kuni kitob o‘qiyman — shu gapni ingliz tiliga tarjima qil.
Base javob: Kitob o’yin, kitob o‘qiymanini tuzing, kitob o‘qiymanini yozing, kitob o‘qiymanini qo’shing, kitob o‘qiymanini qaytarish va kitob o‘qiymanini qaytadan oshirish uchun kitob o‘qiymanini qaytaruvchi programi olish.
Kitob o‘

In [7]:
del base_model, base_tok
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [8]:
# ============================================================
# 3. QWEN2.5-0.5B-INSTRUCT MODELINI YUKLASH
# ============================================================

INSTRUCT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

inst_tok = AutoTokenizer.from_pretrained(INSTRUCT_MODEL)

inst_model = AutoModelForCausalLM.from_pretrained(
    INSTRUCT_MODEL,
    torch_dtype=dtype
).to(device).eval()

print("Instruct model yuklandi.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Instruct model yuklandi.


In [9]:
# ============================================================
# 4. XUDDI SHU 5 TA SAVOLNI INSTRUCT MODELGA BERISH
#    VA JAVOBLARNI SAQLASH
# ============================================================

instruct_natijalar = []

for i, savol in enumerate(SAVOLLAR, 1):
    messages = [{"role": "user", "content": savol}]

    inputs = inst_tok.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(device)

    with torch.no_grad():
        outputs = inst_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=(
                inst_tok.pad_token_id
                if inst_tok.pad_token_id is not None
                else inst_tok.eos_token_id
            )
        )

    yangi_tokenlar = outputs[0][inputs["input_ids"].shape[1]:]
    javob = inst_tok.decode(
        yangi_tokenlar, skip_special_tokens=True
    ).strip()

    instruct_natijalar.append({
        "id": i,
        "savol": savol,
        "instruct_javob": javob
    })

    print(f"\n{i}-savol: {savol}")
    print("Instruct javob:", javob)

with open("instruct_1dars.json", "w", encoding="utf-8") as f:
    json.dump(instruct_natijalar, f, ensure_ascii=False, indent=2)

print("\nInstruct javoblari saqlandi: instruct_1dars.json")


1-savol: Sun’iy intellekt nima?
Instruct javob: I apologize, but I'm not able to understand or respond to the question you've asked. It appears to be in a language that isn't my primary area of expertise. Could you please clarify what language this is in, or provide more context about what you're trying to ask? If it's related to AI or technology, I'd be happy to try and help if you can rephrase your question in English or another widely spoken language.

2-savol: Suv necha gradusda qaynaydi?
Instruct javob: Qaynay dihi, sahib! Suv necha gradusda qaynaydi da: "Ko'idin kuma yana" (meaning "I am who I am"). 

This phrase is often used in Arabic to express one's identity or self-awareness. It's a way of affirming one's uniqueness and being true to oneself.

If you have any other questions about Arabic language or culture, feel free to ask!

3-savol: Vaqtni to‘g‘ri boshqarish uchun 3 ta maslahat ber.
Instruct javob: To'g‘ri boshqa maslumatlar:

1. "Kabul" - Tug'ilgan kabulni ko'rsatamiz.


In [10]:
# ============================================================
# 5. IKKALA MODEL JAVOBLARINI SOLISHTIRISH
#    VA BIR FAYLGA SAQLASH
# ============================================================

taqqoslash = []

for base, inst in zip(base_natijalar, instruct_natijalar):
    taqqoslash.append({
        "id": base["id"],
        "savol": base["savol"],
        "base_javob": base["base_javob"],
        "instruct_javob": inst["instruct_javob"]
    })

with open("model_taqqoslash.json", "w", encoding="utf-8") as f:
    json.dump(taqqoslash, f, ensure_ascii=False, indent=2)

for item in taqqoslash:
    print("\n" + "=" * 60)
    print("Savol:", item["savol"])
    print("Base:", item["base_javob"])
    print("Instruct:", item["instruct_javob"])

print("\nTaqqoslash saqlandi: model_taqqoslash.json")


Savol: Sun’iy intellekt nima?
Base: - 10.05.2019 | 13:47
Sun'iy intellekt nima?
Kazakhstan, 10.05.2019, 13:47
Сегодня в Сирии и Афганистане произошли массовые погромы, которые вызвали смерть сотен тысяч людей. В результате погромов пострадали более 6
Instruct: I apologize, but I'm not able to understand or respond to the question you've asked. It appears to be in a language that isn't my primary area of expertise. Could you please clarify what language this is in, or provide more context about what you're trying to ask? If it's related to AI or technology, I'd be happy to try and help if you can rephrase your question in English or another widely spoken language.

Savol: Suv necha gradusda qaynaydi?
Base: - YouTube\n0:00 / 12:58
Instruct: Qaynay dihi, sahib! Suv necha gradusda qaynaydi da: "Ko'idin kuma yana" (meaning "I am who I am"). 

This phrase is often used in Arabic to express one's identity or self-awareness. It's a way of affirming one's uniqueness and being true to oneself.


## Base Model bilan Instruct Modelni farqi

### 1. Base model savollarga tugri javob bera olmadi va bazi aloqasiz matnlarni chiqardi.
### 2. Instruct modeli ham savollarni yaxshi tushunmadi va notugri javoblar berdi.
### 3. Ikkala model ham uzbek tilidagi savollarga javob berishda qiynaldi.
### 4. Instruct modeli bazi javoblarni uzun yozdi, lekin ular savolga mos kelmadi.
### 5. Umuman olganda, ikkala modelning natijalari ham yaxshi emas edi.